# iSeg-2017 — U-Net 2.5D frugal

Segmentation LCR / substance grise / substance blanche sur IRM T1-T2 de nourrissons de 6 mois.

Ce notebook n'**orchestre** que des appels au paquet `iseg/`, versionné dans git.

**Avant de lancer** : Exécution → Modifier le type d'exécution → **GPU T4**.

## 1. Code, cache et dépendances

Envoie `iseg-colab.zip` (code + cache prétraité, 26 Mo). Le cache évite d'avoir à
transférer les 361 Mo d'IRM brutes — `build_cache` saute les sujets déjà en cache
sans jamais ouvrir les `.hdr/.img`.

In [ ]:
from google.colab import files
files.upload()          # choisis iseg-colab.zip

!mkdir -p /content/repo && unzip -q -o iseg-colab.zip -d /content/repo
%cd /content/repo
!pip -q install nibabel onnx onnxruntime onnxscript

import torch, os
print('GPU  :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN — passer en T4')
print('cache:', len(os.listdir('cache')), 'sujets')

## 2. Entraînement

8 sujets pour apprendre, les sujets 1 et 2 mis de côté pour mesurer. Le Dice 3D sur
ces deux sujets s'affiche toutes les 5 époques ; les poids du meilleur passage sont
écrits dans `runs/<variante>.pt`.

Compter ~10 min sur T4.

In [ ]:
!python -m iseg.train --variant separable --epochs 60

Les deux autres variantes, si tu veux comparer :

| variante | paramètres | int8 | Dice |
|---|---|---|---|
| `standard` | 1 943 636 | 1,89 Mo | 0,8927 |
| `separable` | 386 782 | 0,43 Mo | 0,8624 |
| `tiny` | 26 974 | 0,07 Mo | 0,8281 |

In [ ]:
# !python -m iseg.train --variant standard --epochs 60
# !python -m iseg.train --variant tiny --epochs 60

## 3. Export et deploiement dans la page web

Produit le `.onnx` float32, sa version int8 (~3,5x plus petite), et copie cette derniere
dans `webdemo/` sous le nom `model-<variante>.onnx`, celui que la page charge.

In [ ]:
!python -m iseg.export --checkpoint runs/separable.pt --deploy webdemo

## 4. Recuperer le site

`webdemo/` est un dossier statique : il se publie tel quel sur GitHub Pages, Vercel ou Netlify.

In [ ]:
!zip -qr webdemo.zip webdemo
from google.colab import files
files.download('webdemo.zip')